# Chapter 16: Handling Missing Data

## 16.1 Trade-offs in Missing Data Conventions

결측 자료를 표현하는 두 가지 주요 방식:
| 방식 | 설명 | 장점 | 단점 |
|------|------|------|------|
| **Mask** | 별도의 Boolean 배렬로 결측 표시 | 정확함 | 추가 메모리 필요 |
| **Sentinel** | 특수 값을 결측으로 간주 (예: -9999, NaN) | 메모리 효율적 | 유효 값 범위가 줄어듬 |

💡 Python에서는 주로 NaN(Not a Number) 을 센티넬 값으로 사용한다.

## 16.2 Missing Data in Pandas

In [21]:
import numpy as np
import pandas as pd

# in NumPy array different NaN and None
vals1 = np.array([1, None, 2, 3])
print(vals1)    # [1 None 2 3] → dtype=object (느림!)

vals2 = np.array([1, np.nan, 2, 3])
print(vals2)    # [ 1. nan  2.  3.] → dtype=float64 (빠름!)

[1 None 2 3]
[ 1. nan  2.  3.]


## 16.3 None as a Sentinel Value

Since None is a Python object, it can only be used with arrays of type=object.
Disadvantage: Operations on object arrays are very slow.

In [22]:
%timeit np.array(1_000_000, dtype=int).sum()

%timeit np.array(1_000_000, dtype=object).sum()

1.09 μs ± 15.7 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
1.46 μs ± 607 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [23]:
# None이 포함된 array 연산 불가
vals1 = np.array([1, None, 2, 3])
# vals1.sum()  # TypeError!

## 16.4 NaN: Missing Numerical Data

NaN is a special value in the IEEE floating-point standard.  
Characteristic: The result of an operation involving NaN is always NaN.

In [24]:
vals2 = np.array([1, np.nan, 3, 4])
print(vals2.sum())
print(vals2.min())
print(vals2.max())

# NaN을 무시하는 Version
print(np.nansum(vals2))
print(np.nanmin(vals2))
print(np.nanmax(vals2))

nan
nan
nan
8.0
1.0
4.0


## 16.5 NaN and None in Pandas

In [25]:
pd.Series([1, np.nan, 2, None])

0    1.0
1    NaN
2    2.0
3    NaN
dtype: float64

타입변환 규칙
| 원본 타입 | NA 포함 시 변환 | NA 센티넬 |
|-----------|----------------|------------|
| **float** | 변화 없음 | `np.nan` |
| **object** | 변화 없음 | `None` 또는 `np.nan` |
| **integer** | `float64`로 변환 | `np.nan` |
| **boolean** | `object`로 변환 | `None` 또는 `np.nan` |

In [26]:
# 정수 배렬에 NaN이 들어가면 float로 변환됨
x = pd.Series(range(2), dtype=int)
x[0] = None
print(x)

0    NaN
1    1.0
dtype: float64


## 16.6 Pandas Nullable Dtypes

Pandas provides a Nullable type, allowing NA to be represented even in integer arrays.

In [27]:
# 일반 정수 배렬 (NA 불가)
pd.Series([1, 2, 3], dtype='int64')

# Nullable 정수 배렬 (NA 가능)
pd.Series([1, np.nan, 2, None, pd.NA], dtype='Int32')   # Int32 = Nullable Integer Type

0       1
1    <NA>
2       2
3    <NA>
4    <NA>
dtype: Int32

Nullable Type list
| 일반 타입 | Nullable 타입 |
|-----------|---------------|
| `int64` | `Int64` (대문자 주의!) |
| `float64` | `Float64` |
| `bool` | `boolean` |
| `str` | `string` |

## 16.7 Operating on Null Values

Missing value handling methods provided by Pandas:
| 메서드 | 설명 |
|--------|------|
| `isnull()` | 결측값이면 `True` 반환 |
| `notnull()` | 결측값이 아니면 `True` 반환 |
| `dropna()` | 결측값이 있는 행/열 제거 |
| `fillna()` | 결측값을 다른 값으로 채움 |

Detecting Null Values

In [28]:
data = pd.Series([1, np.nan, 'hello', None])

print(data.isnull())

print(data.notnull())

0    False
1     True
2    False
3     True
dtype: bool
0     True
1    False
2     True
3    False
dtype: bool


****Dropping Null Values 🔥important****

In [29]:
df = pd.DataFrame([[1, np.nan, 2],
                   [2, 3, 5],
                   [np.nan, 4, 6]])
print(df)

     0    1  2
0  1.0  NaN  2
1  2.0  3.0  5
2  NaN  4.0  6


* Default: Remove rows with at least one missing value

In [30]:
print(df.dropna())

     0    1  2
1  2.0  3.0  5


* Remove column direction (axis=1)

In [31]:
print(df.dropna(axis='columns'))

   2
0  2
1  5
2  6


* how='all' - Remove only cases where all are missing

In [32]:
df[3] = np.nan
print(df)

print(df.dropna(axis='columns', how='all'))

     0    1  2   3
0  1.0  NaN  2 NaN
1  2.0  3.0  5 NaN
2  NaN  4.0  6 NaN
     0    1  2
0  1.0  NaN  2
1  2.0  3.0  5
2  NaN  4.0  6


* thresh - Specify minimum number of non-null values

In [33]:
print(df.dropna(axis='rows', thresh=3))

     0    1  2   3
1  2.0  3.0  5 NaN


****Filling Null Values 🔥 important!****

In [34]:
data = pd.Series([1, np.nan, 2, None, 3],
                 index=list('abcde'), dtype='Int32')
print(data)

a       1
b    <NA>
c       2
d    <NA>
e       3
dtype: Int32


* filling single value

In [35]:
print(data.fillna(0))

a    1
b    0
c    2
d    0
e    3
dtype: Int32


* Forward fill

In [37]:
print(data.ffill())

a    1
b    1
c    2
d    2
e    3
dtype: Int32


* backward fill

In [38]:
print(data.bfill())

a    1
b    2
c    2
d    3
e    3
dtype: Int32


* Orienting a DataFrame

In [40]:
df = pd.DataFrame([[1, np.nan, 2],
                   [2, 3, 5],
                   [np.nan, 4, 6]])
df[3] = np.nan

print(df.ffill(axis=1))

     0    1    2    3
0  1.0  1.0  2.0  2.0
1  2.0  3.0  5.0  5.0
2  NaN  4.0  6.0  6.0
